# Soft Actor-Critic (SAC) — Ablation Study & Empirical Analysis

This notebook provides systematic analysis and visualization of our SAC implementation and empirical ablation experiments.

### Ablation Axes Explored:
1. **Entropy Temperature ($\alpha$) Tuning**: Fixed temperature values ($\alpha \in \{0.0, 0.05, 0.2, 1.0\}$) vs. Automatically tuned $\alpha$ via dual gradient descent (Haarnoja et al., 2018b).
2. **Stochastic vs. Deterministic Evaluation**: Comparing stochastic evaluation actions (sampled from $\pi$) vs. deterministic actions (mean $\tanh(\mu)$).
3. **Twin Q vs. Single Q Critic**: Verifying whether the Clipped Double-Q trick prevents value overestimation bias and maintains policy stability.
4. **Reward Scaling Sensitivity**: Evaluating the sensitivity of fixed temperature ($\alpha = 0.2$) across reward magnitudes ($s \in \{1, 5, 10, 20\}$).
5. **Target Entropy Heuristic**: Examining policy exploration across target entropy scalings $\mathcal{H}_{\text{target}} = c \cdot (-|\mathcal{A}|)$ with $c \in \{0.5, 1.0, 1.5, 2.0\}$.
6. **Safety Constraint Enforcement**: Evaluating cost-penalized SAC in Constrained MDPs (`SafetyAntVelocity-v1`) across penalty strengths $\lambda$.

In [ ]:
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from scripts.plot_results import (
    load_all_logs,
    plot_ablation_group,
    plot_alpha_trajectory,
    plot_q_overestimation,
    plot_baseline_comparison,
    generate_summary_table,
)

log_dir = os.path.join("..", "results", "logs")
logs = load_all_logs(log_dir)
print(f"Loaded {len(logs)} experiment log file(s) from {log_dir}")
for k in sorted(logs.keys()):
    print(f"  • {k}")

## 1. Baseline Comparison (Stable-Baselines3 vs. Our SAC)
Validates that our from-scratch SAC matches reference implementations in learning dynamics and performance.

In [ ]:
plot_baseline_comparison(logs, plot_dir="../results/plots")
baseline_img = "../results/plots/baseline_comparison.png"
if os.path.exists(baseline_img):
    from IPython.display import Image, display
    display(Image(baseline_img))
else:
    print("Baseline plot not yet generated or missing logs.")

## 2. Axis 1: Entropy Temperature ($\alpha$) Tuning
Investigates the trade-off between policy entropy (exploration) and expected return (exploitation), comparing fixed temperatures against automatic temperature tuning.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="alpha_",
    title="SAC Ablation Study: Temperature (α) Tuning",
    output_name="ablation_alpha_sweep.png",
    label_fn=lambda name, data: "Auto-tuned α" if "auto" in name.lower() else f"Fixed α = {name.replace('alpha_', '')}",
    plot_dir="../results/plots",
)
plot_alpha_trajectory(logs, plot_dir="../results/plots")

alpha_img = "../results/plots/ablation_alpha_sweep.png"
alpha_dyn_img = "../results/plots/ablation_alpha_dynamics.png"
from IPython.display import Image, display
for img in [alpha_img, alpha_dyn_img]:
    if os.path.exists(img):
        display(Image(img))

## 3. Axis 2: Twin Q vs. Single Q Critic
Evaluates the impact of Clipped Double Q-learning. In continuous action spaces, single Q-networks suffer from severe positive maximization bias.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="use_twin_q_",
    title="SAC Ablation Study: Twin Q vs. Single Q Critic",
    output_name="ablation_twin_vs_single_q.png",
    label_fn=lambda name, data: "Twin Q (Clipped Double-Q)" if "true" in name.lower() else "Single Q Critic",
    plot_dir="../results/plots",
)
plot_q_overestimation(logs, plot_dir="../results/plots")

for img in ["../results/plots/ablation_twin_vs_single_q.png", "../results/plots/ablation_q_overestimation.png"]:
    if os.path.exists(img):
        display(Image(img))

## 4. Axis 3: Stochastic vs. Deterministic Policy Evaluation
Compares sampling actions from the stochastic policy during evaluation versus taking the deterministic policy mode $\tanh(\mu(s))$.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="deterministic_eval_",
    title="SAC Ablation Study: Stochastic vs. Deterministic Evaluation",
    output_name="ablation_stochastic_vs_det.png",
    label_fn=lambda name, data: "Deterministic Policy (Mean)" if "true" in name.lower() else "Stochastic Policy (Sampled)",
    plot_dir="../results/plots",
)
det_img = "../results/plots/ablation_stochastic_vs_det.png"
if os.path.exists(det_img):
    display(Image(det_img))

## 5. Axis 4: Reward Scaling Sensitivity
Tests the hypothesis that fixed-temperature SAC is sensitive to the scale of the environment rewards, necessitating manual hyperparameter tuning when auto-tuning is disabled.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="reward_scale_",
    title="SAC Ablation Study: Reward Scaling Sensitivity (Fixed α = 0.2)",
    output_name="ablation_reward_scaling.png",
    label_fn=lambda name, data: f"Reward Scale = {name.replace('reward_scale_', '')}",
    plot_dir="../results/plots",
)
rew_img = "../results/plots/ablation_reward_scaling.png"
if os.path.exists(rew_img):
    display(Image(rew_img))

## 6. Axis 5: Target Entropy Heuristic Sweep
Tests sensitivity to the heuristic target entropy $\mathcal{H}_{\text{target}} = -|\mathcal{A}|$ across fractional and multiple values.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="target_entropy_",
    title="SAC Ablation Study: Target Entropy Heuristic Scaling",
    output_name="ablation_target_entropy.png",
    label_fn=lambda name, data: f"H_target = {name.replace('target_entropy_', '')}",
    plot_dir="../results/plots",
)
ent_img = "../results/plots/ablation_target_entropy.png"
if os.path.exists(ent_img):
    display(Image(ent_img))

## 7. Axis 6: Safety Cost Penalization (Constrained MDP)
Evaluates the trade-off between task reward and constraint violation costs on `SafetyAntVelocity-v1`.

In [ ]:
plot_ablation_group(
    logs,
    filter_prefix="cost_penalty_",
    title="SAC Ablation Study: Safety Cost Penalty (SafetyAntVelocity-v1)",
    output_name="ablation_safety_cost_penalty.png",
    label_fn=lambda name, data: f"Penalty λ = {name.replace('cost_penalty_', '')}",
    plot_dir="../results/plots",
)
safe_img = "../results/plots/ablation_safety_cost_penalty.png"
if os.path.exists(safe_img):
    display(Image(safe_img))

## 8. Summary Table of Empirical Results
Generates a comparative table across all ablation conditions.

In [ ]:
generate_summary_table(logs, output_dir="../results")

records = []
for name, data in sorted(logs.items()):
    if name.startswith("sb3_baseline_"):
        continue
    cfg = data.get("config", {})
    history = data.get("history", {})
    means = history.get("eval_reward_mean", [])
    stds = history.get("eval_reward_std", [])
    costs = history.get("eval_cost_mean", [])
    steps = history.get("steps", [])
    if means:
        records.append({
            "Experiment": name,
            "Environment": cfg.get("env_id", "-"),
            "Timesteps": steps[-1] if steps else 0,
            "Final Return": f"{means[-1]:.2f} ± {stds[-1]:.2f}",
            "Max Return": f"{max(means):.2f}",
            "Safety Cost": f"{costs[-1]:.1f}" if costs and any(c > 0 for c in costs) else "-",
        })

if records:
    df = pd.DataFrame(records)
    display(df)
else:
    print("No completed experiment logs found yet.")